In [107]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [108]:
data = pd.read_parquet("chicago_sales_with_crime.parquet")

In [109]:
data.shape

(56629, 85)

In [110]:
data.columns

Index(['pin', 'year', 'township_code', 'neighborhood_code', 'class',
       'sale_date', 'sale_price', 'is_multisale',
       'sale_filter_same_sale_within_365', 'sale_filter_less_than_10k',
       'sale_filter_deed_type', 'year_built', 'building_sqft', 'land_sqft',
       'num_bedrooms', 'num_rooms', 'num_full_baths', 'num_half_baths',
       'num_fireplaces', 'type_of_residence', 'construction_quality',
       'attic_finish', 'garage_size', 'basement_type', 'ext_wall_material',
       'central_heating', 'repair_condition', 'basement_finish',
       'single_v_multi_family', 'central_air', 'pin10', 'zip_code',
       'longitude', 'latitude', 'centroid_x_crs_3435', 'centroid_y_crs_3435',
       'community_area', 'flood_fema_sfha', 'flood_fs_factor',
       'airport_noise_dnl', 'school_elementary_district_name',
       'school_secondary_district_name', 'cmap_walkability_total_score',
       'census_acs5_tract_geoid', 'num_pin_in_half_mile',
       'num_bus_stop_in_half_mile',
       'num

In [111]:
data["type_of_residence"].unique()

['1.5 Story', '2 Story', '1 Story', '3 Story +', 'Split Level']
Categories (5, object): ['1 Story', '1.5 Story', '2 Story', '3 Story +', 'Split Level']

In [112]:
features_to_exclude = ["pin","is_multisale", "sale_filter_same_sale_within_365", "sale_filter_less_than_10k", "single_v_multi_family", "zip_code",
                      "longitude", "latitude", "sale_date_parsed", "class", 'centroid_x_crs_3435', 'centroid_y_crs_3435', 
                       "school_elementary_district_name", "school_secondary_district_name", "township_code",  "neighborhood_code"]
#excluding pin is clear
#excluding is_multisale since it is always false
#excluding sale_filter_same_sale_within_365 because it is always false
#excluding sale_filter_less_than_10k because it is always false
#excluding single_v_multi_family since we are only looking at single family homes
#excluding zip_code,longitude, and latitude since we want all of the location data to be coming from the community area codes

In [113]:
cat_features = ["community_area", "type_of_residence", "sale_date", "construction_quality", 
                "attic_finish", "garage_size", "basement_type", "ext_wall_material", "central_heating", "repair_condition",
                "basement_finish", "central_air", "flood_fema_sfha", "sale_filter_deed_type"
               ]

In [114]:
included_features = []
for c in data.columns:
    if c in features_to_exclude:
        continue
    included_features.append(c)

In [115]:
data = data[included_features]

In [116]:
data = pd.get_dummies(data, columns = cat_features, dtype=int)

In [117]:
data.head()

,year,sale_price,year_built,building_sqft,land_sqft,num_bedrooms,num_rooms,num_full_baths,num_half_baths,num_fireplaces,...,repair_condition_Average,repair_condition_Below Average,basement_finish_Apartment,basement_finish_Formal Rec Room,basement_finish_Unfinished,central_air_Central A/C,central_air_No Central A/C,flood_fema_sfha_False,flood_fema_sfha_True,sale_filter_deed_type_False
0,2021,75750,1903.0,1212.0,3055.0,3.0,8.0,1.0,1.0,0.0,...,1,0,0,1,0,0,1,1,0,1
1,2021,300000,1893.0,2112.0,4450.0,4.0,6.0,2.0,0.0,0.0,...,1,0,0,0,1,0,1,1,0,1
2,2021,2310000,1898.0,2844.0,3800.0,5.0,9.0,3.0,0.0,0.0,...,0,0,0,1,0,1,0,1,0,1
3,2021,222500,1952.0,840.0,3720.0,2.0,4.0,1.0,0.0,0.0,...,1,0,0,0,1,0,1,1,0,1
4,2021,1015000,2000.0,2675.0,2400.0,6.0,12.0,6.0,0.0,1.0,...,1,0,0,1,0,1,0,1,0,1


In [118]:
from sklearn.model_selection import train_test_split

In [119]:
data=data.dropna()
data.shape

(36512, 1948)

In [120]:
y = data["sale_price"]
features = list(data.columns)
features.remove("sale_price")
X = data[features]

In [121]:
from sklearn.model_selection import KFold, cross_val_score, cross_validate

In [122]:
# 1. Build a pipeline: scale features, then fit Ridge
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=1.0, random_state=42))
])

# 2. Set up cross-validation splitting strategy
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# 3. Run cross-validation with multiple scoring metrics
scoring = {
    "r2": "r2",
    "neg_mae": "neg_mean_absolute_error",
    "neg_rmse": "neg_root_mean_squared_error"
}

results = cross_validate(
    pipeline, X, y,
    cv=cv,
    scoring=scoring,
    return_train_score=True
)

# 4. Summarize results
print("R²:    mean = {:.4f}, std = {:.4f}".format(
    results["test_r2"].mean(), results["test_r2"].std()))
print("MAE:   mean = {:.4f}, std = {:.4f}".format(
    -results["test_neg_mae"].mean(), results["test_neg_mae"].std()))
print("RMSE:  mean = {:.4f}, std = {:.4f}".format(
    -results["test_neg_rmse"].mean(), results["test_neg_rmse"].std()))

# Optional: compare to train scores to check for overfitting
print("\nTrain R² (for comparison): {:.4f}".format(results["train_r2"].mean()))

R²:    mean = 0.7779, std = 0.0190
MAE:   mean = 136773.8825, std = 2119.1612
RMSE:  mean = 227183.6648, std = 22451.6451

Train R² (for comparison): 0.8003


In [123]:
# Fit the pipeline on the entire training data
pipeline.fit(X, y)

# Extract the fitted Ridge model and coefficients
ridge_model = pipeline.named_steps["ridge"]
coefficients = ridge_model.coef_
intercept = ridge_model.intercept_

# Put coefficients into a readable dataframe, sorted by magnitude
coef_df = pd.DataFrame({
    "feature": X.columns,
    "coefficient": coefficients
}).sort_values(by="coefficient", key=abs, ascending=False)

print("Intercept:", intercept)
print("\nCoefficients (sorted by absolute magnitude):")
print(coef_df.to_string(index=False))

Intercept: 430400.24460453365

Coefficients (sorted by absolute magnitude):
                                  feature   coefficient
                            building_sqft 227263.755601
                         community_area_7 100546.443688
                       garage_size_2 cars -89580.011922
                       garage_size_0 cars -74374.961391
                       garage_size_1 cars -70269.395697
                       basement_type_Slab -60691.687421
                        community_area_24  57225.935816
                           num_full_baths  53178.799145
                         community_area_6  51769.001755
                         community_area_5  51194.042614
                         community_area_8  50071.695813
                        community_area_22  48306.845904
num_foreclosure_in_half_mile_past_5_years -44497.480349
                     garage_size_1.5 cars -43103.775555
               total_homicides_since_2020 -42643.997779
num_foreclosure_per_1000_pin

In [124]:
community_coef_df = coef_df[coef_df["feature"].str.startswith("community_area_")]

print(community_coef_df.to_string(index=False))

          feature   coefficient
 community_area_7 100546.443688
community_area_24  57225.935816
 community_area_6  51769.001755
 community_area_5  51194.042614
 community_area_8  50071.695813
community_area_22  48306.845904
community_area_49 -38172.639938
community_area_48 -32095.606966
community_area_53 -31061.064138
community_area_44 -28840.673324
community_area_46 -26481.822535
community_area_69 -26269.910985
community_area_45 -25110.514472
community_area_51 -22753.105039
community_area_56  22617.876872
community_area_73 -22450.236135
community_area_72 -22191.584007
community_area_50 -21691.992870
community_area_70 -21304.268844
community_area_75 -20958.635760
community_area_38 -18282.885515
 community_area_4  17849.168221
community_area_71 -17471.720448
community_area_47 -17246.417982
community_area_21  16591.274897
community_area_28  16491.421857
community_area_33  15286.529003
community_area_42 -14273.781600
community_area_25  13735.559403
community_area_66 -13457.934260
communit